In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
%matplotlib inline

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df = df.copy()

## Data Preparation

In [6]:
df.columns = df.columns.str.lower().str.replace(" ","-")

In [7]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

In [10]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [16]:
df.isnull().sum()

lead_source                   0
industry                      0
employment_status             0
location                      0
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [12]:
categorical = ['lead_source','industry','employment_status','location']
numerical = [i for i in df.columns if i not in categorical and i not in ['converted']]

In [15]:
df[categorical] = df[categorical].fillna("NA")

In [17]:
df[numerical] = df[numerical].fillna(0.0)

In [18]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Q1
What is the most frequent observation (mode) for the column industry?

In [21]:
df.industry.value_counts()
# tech is the most frequent

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## Q2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

In [24]:
df[numerical].corr()
# lead score and interaction_count has the biggest corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


Split the data

In [27]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

## Q3

Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).

In [28]:
from sklearn.metrics import mutual_info_score

In [31]:
def mutual_info_convert_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [33]:
df_full_train[categorical].apply(mutual_info_convert_score)
# Lead_source has the biggest mutual info score

lead_source          0.034761
industry             0.002898
employment_status    0.020313
location             0.001050
dtype: float64

## Q4

In [58]:
from sklearn import set_config
set_config(display="text")
from sklearn.feature_extraction import DictVectorizer

In [59]:
dicts = df_train[categorical+numerical].to_dict(orient='records')
val_dicts = df_val[categorical+numerical].to_dict(orient='records')

In [60]:
dv = DictVectorizer(sparse=False)

In [61]:
X_train = dv.fit_transform(dicts)
X_val = dv.transform(val_dicts)

In [63]:
y_train = df_train.converted
y_val = df_val.converted

In [64]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [74]:
y_pred = model.predict_proba(X_val)[:,1]
convert_decision = (y_pred >= 0.5)
(convert_decision == y_val).mean().round(2).tolist()

0.64

## Q5

In [77]:
base = (convert_decision == y_val).mean().tolist()
base

0.645

In [79]:
def train_small(lst):
    dicts_small = df_train[lst].to_dict(orient='records')
    val_dicts_small = df_val[lst].to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    
    X_train = dv.fit_transform(dicts_small)
    X_val = dv.transform(val_dicts_small)
    y_train = df_train.converted
    y_val = df_val.converted

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train,y_train)

    y_pred = model.predict_proba(X_val)[:,1]
    convert_decision = (y_pred >= 0.5)
    result =  (convert_decision == y_val).mean().tolist()
    return base - result

In [85]:
options = categorical+numerical
for col in options:
    new_list = [i for i in options if i != col]
    diff = train_small(new_list)
    print(col," : ",diff)

lead_source  :  0.0030000000000000027
industry  :  0.0
employment_status  :  0.0
location  :  0.0
annual_income  :  -0.07899999999999996
number_of_courses_viewed  :  0.0020000000000000018
interaction_count  :  0.04400000000000004
lead_score  :  0.0010000000000000009


## Q6

In [89]:
C = [0.000001, 0.00001, 0.0001, 0.001]

In [92]:
def train_logi(lst,c):
    dicts = df_train[lst].to_dict(orient='records')
    val_dicts = df_val[lst].to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    
    X_train = dv.fit_transform(dicts)
    X_val = dv.transform(val_dicts)
    y_train = df_train.converted
    y_val = df_val.converted

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train,y_train)

    y_pred = model.predict_proba(X_val)[:,1]
    convert_decision = (y_pred >= 0.5)
    result =  (convert_decision == y_val).mean().tolist()
    return result

In [93]:
for val in C:
    result = train_logi(options,val)
    print(val," : ",result)

1e-06  :  0.598
1e-05  :  0.598
0.0001  :  0.613
0.001  :  0.645
